# Review Gap EDA

This notebook analyzes `data/Description_PROC.csv` and `data/Reviews_PROC.csv` to find missing coverage and information gaps between listing content and what guests actually talk about.

It combines:

- basic EDA and missingness profiling
- lexical facet detection with hand-built regexes
- semantic facet detection with an LSA-based semantic space
- gap analysis between listing facets and review coverage
- unsupervised topic clustering to discover themes outside the curated facets
- lightweight supervised ML checks to estimate how predictable review quality signals are from text

The notebook is designed to run locally without external APIs.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
TODAY = pd.Timestamp.utcnow().tz_localize(None).normalize()

FACET_LEXICON = {
    "pet": r"\b(pet|pets|dog|dogs|cat|cats|animal|service animal)\b",
    "check_in": r"\b(check.?in|checkin|front desk|arrival|arrived|reception|lobby|key|keycard)\b",
    "check_out": r"\b(check.?out|checkout|depart|departure|late check|early check)\b",
    "amenities_pool": r"\b(pool|swimming|hot tub|jacuzzi|spa)\b",
    "amenities_wifi": r"\b(wifi|wi-fi|internet|connection|bandwidth)\b",
    "amenities_breakfast": r"\b(breakfast|buffet|continental|morning meal)\b",
    "amenities_parking": r"\b(parking|valet|garage|lot)\b",
    "amenities_gym": r"\b(gym|fitness|workout|exercise)\b",
    "children_extra_bed": r"\b(kid|kids|child|children|crib|cot|extra bed|rollaway|family)\b",
    "know_before_you_go": r"\b(construction|renovation|noise|noisy|loud|fee|charge|deposit|unexpected)\b",
}

FACET_PROTOTYPES = {
    "pet": [
        "The hotel allows pets and dogs.",
        "There were surprise pet fees or pet restrictions.",
        "This stay was not pet friendly.",
    ],
    "check_in": [
        "Check-in at the front desk was smooth and fast.",
        "Check-in was slow and the room was not ready.",
        "We had issues getting keys on arrival.",
    ],
    "check_out": [
        "Check-out was easy and quick.",
        "Late checkout was denied or expensive.",
        "Check-out fees were a surprise.",
    ],
    "amenities_pool": [
        "The pool and hot tub were open, clean, and usable.",
        "The pool was closed, dirty, or disappointing.",
        "We enjoyed the swimming pool area.",
    ],
    "amenities_wifi": [
        "The Wi-Fi was fast and reliable.",
        "The internet connection was slow or kept dropping.",
        "Wi-Fi worked well for business travel.",
    ],
    "amenities_breakfast": [
        "Breakfast was included and good.",
        "Breakfast options were limited or disappointing.",
        "The buffet matched the listing.",
    ],
    "amenities_parking": [
        "Parking was easy and available on site.",
        "Parking was hard to find or expensive.",
        "Valet and parking access were convenient.",
    ],
    "amenities_gym": [
        "The gym or fitness center had usable equipment.",
        "The gym was closed, broken, or too small.",
        "Workout facilities were available.",
    ],
    "children_extra_bed": [
        "The stay was family friendly and cribs were available.",
        "There was no extra bed or rollaway when needed.",
        "The hotel worked well for children.",
    ],
    "know_before_you_go": [
        "There were hidden fees, restrictions, or deposits.",
        "Construction or noise affected the stay.",
        "Important restrictions were not clear in advance.",
    ],
}

LISTING_FACET_FIELDS = {
    "pet": ["pet_policy"],
    "check_in": ["check_in_start_time", "check_in_end_time", "check_in_instructions"],
    "check_out": ["check_out_time", "check_out_policy"],
    "amenities_pool": ["property_amenity_outdoor", "property_amenity_spa", "popular_amenities_list"],
    "amenities_wifi": ["property_amenity_internet", "popular_amenities_list"],
    "amenities_breakfast": ["property_amenity_food_and_drink", "popular_amenities_list"],
    "amenities_parking": ["property_amenity_parking", "popular_amenities_list"],
    "amenities_gym": ["property_amenity_more", "property_amenity_things_to_do"],
    "children_extra_bed": ["children_and_extra_bed_policy", "property_amenity_family_friendly"],
    "know_before_you_go": ["know_before_you_go"],
}

POS_CUES = {
    "clean", "great", "good", "excellent", "friendly", "helpful", "nice", "comfortable",
    "perfect", "amazing", "love", "loved", "open", "smooth", "easy", "fast", "quiet", "spotless",
}
NEG_CUES = {
    "dirty", "bad", "terrible", "rude", "unfriendly", "slow", "noisy", "broken", "closed",
    "stain", "smelly", "smell", "odor", "filthy", "mold", "disappointed", "awful", "horrible",
    "unexpected", "fee", "charge",
}

FACETS = list(FACET_LEXICON)


def parse_rating_map(value):
    if not isinstance(value, str) or not value.strip():
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {}


def build_listing_docs(desc_df):
    docs = {}
    for _, row in desc_df.iterrows():
        pid = row["eg_property_id"]
        docs[pid] = {}
        for facet, cols in LISTING_FACET_FIELDS.items():
            parts = []
            for col in cols:
                value = row.get(col)
                if isinstance(value, str) and value.strip() and value.strip() != "[]":
                    parts.append(value.strip())
            docs[pid][facet] = " ".join(parts)
    return docs


def lexical_mentions(rev_df):
    text = (rev_df["review_title"] + " " + rev_df["review_text"]).str.lower()
    hits = {facet: text.str.contains(pattern, regex=True, na=False).astype(int) for facet, pattern in FACET_LEXICON.items()}
    out = pd.DataFrame(hits, index=rev_df.index)
    out.insert(0, "eg_property_id", rev_df["eg_property_id"].values)
    out.insert(1, "acquisition_date", rev_df["acquisition_date"].values)
    return out


def fit_semantic_space(rev_df, listing_docs):
    review_texts = rev_df["full_text"].fillna("").tolist()
    listing_texts = [doc for facet_docs in listing_docs.values() for doc in facet_docs.values() if doc]
    prototype_texts = [text for texts in FACET_PROTOTYPES.values() for text in texts]
    corpus = review_texts + listing_texts + prototype_texts

    vectorizer = TfidfVectorizer(
        stop_words="english",
        min_df=2,
        max_df=0.9,
        ngram_range=(1, 2),
        strip_accents="unicode",
        max_features=30000,
    )
    X = vectorizer.fit_transform(corpus)
    n_components = min(150, max(10, X.shape[1] - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    Z = svd.fit_transform(X)

    n_reviews = len(review_texts)
    n_listing = len(listing_texts)
    review_vecs = Z[:n_reviews]
    listing_vecs = Z[n_reviews:n_reviews + n_listing]
    proto_vecs = Z[n_reviews + n_listing:]

    listing_lookup = {}
    listing_index = 0
    for pid, facet_docs in listing_docs.items():
        listing_lookup[pid] = {}
        for facet, doc in facet_docs.items():
            if doc:
                listing_lookup[pid][facet] = listing_vecs[listing_index]
                listing_index += 1
            else:
                listing_lookup[pid][facet] = None

    proto_lookup = {}
    proto_index = 0
    for facet, texts in FACET_PROTOTYPES.items():
        proto_lookup[facet] = proto_vecs[proto_index:proto_index + len(texts)]
        proto_index += len(texts)

    semantic_scores = pd.DataFrame(index=rev_df.index)
    for facet, proto_matrix in proto_lookup.items():
        semantic_scores[facet] = cosine_similarity(review_vecs, proto_matrix).max(axis=1)

    thresholds = {}
    for facet in FACETS:
        lexical = rev_df[f"lex_{facet}"].astype(int)
        grid = np.arange(0.15, 0.71, 0.05)
        best_threshold = 0.30
        best_score = -1.0
        for threshold in grid:
            semantic_hit = (semantic_scores[facet] >= threshold).astype(int)
            both = int(((semantic_hit == 1) & (lexical == 1)).sum())
            union = int(((semantic_hit == 1) | (lexical == 1)).sum())
            jaccard = both / union if union else 0.0
            if (jaccard > best_score) or (jaccard == best_score and threshold > best_threshold):
                best_score = jaccard
                best_threshold = float(threshold)
        thresholds[facet] = best_threshold

    semantic_hits = pd.DataFrame({facet: (semantic_scores[facet] >= thresholds[facet]).astype(int) for facet in FACETS}, index=rev_df.index)
    return review_vecs, listing_lookup, proto_lookup, semantic_scores, semantic_hits, thresholds


def evaluate_binary(df, target_col, splitter, use_grouped=False):
    rows = []
    for model_name, model in {
        "dummy": DummyClassifier(strategy="prior"),
        "tfidf_logreg": Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english", min_df=3, max_df=0.9, ngram_range=(1, 2), max_features=25000)),
            ("model", LogisticRegression(max_iter=500, class_weight="balanced")),
        ]),
    }.items():
        aucs, aps, f1s = [], [], []
        split_iter = splitter.split(df, df[target_col], groups=df["eg_property_id"] if use_grouped else None)
        for train_idx, test_idx in split_iter:
            train = df.iloc[train_idx]
            test = df.iloc[test_idx]
            if train[target_col].nunique() < 2 or test[target_col].nunique() < 2:
                continue
            model.fit(train["full_text"], train[target_col])
            prob = model.predict_proba(test["full_text"])[:, 1]
            pred = (prob >= 0.5).astype(int)
            aucs.append(roc_auc_score(test[target_col], prob))
            aps.append(average_precision_score(test[target_col], prob))
            f1s.append(f1_score(test[target_col], pred, zero_division=0))
        if aucs:
            rows.append({
                "model": model_name,
                "roc_auc": np.mean(aucs),
                "avg_precision": np.mean(aps),
                "f1": np.mean(f1s),
                "folds": len(aucs),
            })
    return pd.DataFrame(rows)


def evaluate_regression(df, target_col, splitter, use_grouped=False):
    rows = []
    for model_name, model in {
        "dummy": DummyRegressor(strategy="mean"),
        "tfidf_ridge": Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english", min_df=3, max_df=0.9, ngram_range=(1, 2), max_features=25000)),
            ("model", Ridge(alpha=1.0)),
        ]),
    }.items():
        maes = []
        split_iter = splitter.split(df, groups=df["eg_property_id"] if use_grouped else None)
        for train_idx, test_idx in split_iter:
            train = df.iloc[train_idx]
            test = df.iloc[test_idx]
            model.fit(train["full_text"], train[target_col])
            pred = np.clip(model.predict(test["full_text"]), 1.0, 5.0)
            maes.append(mean_absolute_error(test[target_col], pred))
        if maes:
            rows.append({"model": model_name, "mae": np.mean(maes), "folds": len(maes)})
    return pd.DataFrame(rows)

print(f"Using data directory: {DATA}")

In [ ]:
desc = pd.read_csv(DATA / "Description_PROC.csv")
rev = pd.read_csv(DATA / "Reviews_PROC.csv")
rev["acquisition_date"] = pd.to_datetime(rev["acquisition_date"], format="%m/%d/%y", errors="coerce")
rev["review_title"] = rev["review_title"].fillna("")
rev["review_text"] = rev["review_text"].fillna("")
rev["full_text"] = (rev["review_title"] + " " + rev["review_text"]).str.strip()
rev["text_len"] = rev["review_text"].str.len()
rev["title_len"] = rev["review_title"].str.len()

rating_df = pd.json_normalize(rev["rating"].apply(parse_rating_map))
rating_df.columns = [f"rating_{c}" for c in rating_df.columns]
rev = pd.concat([rev.drop(columns=["rating"]), rating_df], axis=1)

profile = pd.Series({
    "properties": desc["eg_property_id"].nunique(),
    "reviews": len(rev),
    "review_date_min": rev["acquisition_date"].min(),
    "review_date_max": rev["acquisition_date"].max(),
    "empty_review_text_pct": round((rev["review_text"].eq("").mean() * 100), 2),
    "empty_review_title_pct": round((rev["review_title"].eq("").mean() * 100), 2),
    "median_review_chars": int(rev["text_len"].median()),
    "mean_review_chars": round(rev["text_len"].mean(), 1),
    "median_reviews_per_property": int(rev.groupby("eg_property_id").size().median()),
    "min_reviews_per_property": int(rev.groupby("eg_property_id").size().min()),
    "max_reviews_per_property": int(rev.groupby("eg_property_id").size().max()),
})

profile.to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

review_counts = rev.groupby("eg_property_id").size().sort_values()
axes[0].barh(review_counts.index.astype(str), review_counts.values, color="#2563eb")
axes[0].set_title("Reviews per property")
axes[0].set_xlabel("count")

rating_cols = [c for c in rev.columns if c.startswith("rating_")]
rating_coverage = (rev[rating_cols].fillna(0) > 0).mean().sort_values()
axes[1].barh([c.replace("rating_", "") for c in rating_coverage.index], rating_coverage.values * 100, color="#7c3aed")
axes[1].set_title("Rating field coverage")
axes[1].set_xlabel("% of reviews with non-zero label")

axes[2].hist(rev.loc[rev["text_len"] > 0, "text_len"], bins=50, color="#059669")
axes[2].set_title("Review text length distribution")
axes[2].set_xlabel("characters")
axes[2].set_ylabel("reviews")

plt.tight_layout()

missing_listing = desc.isna().mean().sort_values(ascending=False).head(12).rename("missing_rate")
missing_listing.to_frame()

In [ ]:
lex = lexical_mentions(rev)
for facet in FACETS:
    rev[f"lex_{facet}"] = lex[facet].values

text_tokens = rev["full_text"].str.lower().str.findall(r"[a-z']+")
rev["pos_terms"] = text_tokens.apply(lambda tokens: sum(token in POS_CUES for token in tokens))
rev["neg_terms"] = text_tokens.apply(lambda tokens: sum(token in NEG_CUES for token in tokens))
rev["neg_heavy"] = (rev["neg_terms"] > rev["pos_terms"]).astype(int)

lexical_coverage = (
    rev[[f"lex_{facet}" for facet in FACETS]]
    .mean()
    .rename(lambda col: col.replace("lex_", ""))
    .sort_values()
    .mul(100)
    .rename("review_mention_pct")
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
colors = ["#dc2626" if x < 10 else "#f59e0b" if x < 25 else "#16a34a" for x in lexical_coverage.values]
axes[0].barh(lexical_coverage.index, lexical_coverage.values, color=colors)
axes[0].set_title("Lexical facet coverage across reviews")
axes[0].set_xlabel("% of reviews mentioning facet")

property_lex = rev.groupby("eg_property_id")[[f"lex_{facet}" for facet in FACETS]].mean()
sns.heatmap(property_lex.rename(columns=lambda c: c.replace("lex_", "")), cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "mention rate"})
axes[1].set_title("Property x facet lexical coverage")
axes[1].set_xlabel("facet")
axes[1].set_ylabel("property")

plt.tight_layout()
lexical_coverage.to_frame()

In [ ]:
listing_docs = build_listing_docs(desc)
listing_presence = pd.DataFrame(
    {
        facet: {pid: int(bool(facet_docs.get(facet, "").strip())) for pid, facet_docs in listing_docs.items()}
        for facet in FACETS
    }
).sort_index()

rows = []
for pid, group in rev.groupby("eg_property_id"):
    for facet in FACETS:
        facet_hits = group[f"lex_{facet}"] == 1
        last_mention = group.loc[facet_hits, "acquisition_date"].max() if facet_hits.any() else pd.NaT
        rows.append(
            {
                "eg_property_id": pid,
                "facet": facet,
                "listing_mentions_facet": int(listing_presence.loc[pid, facet]),
                "review_mention_rate_lex": group[f"lex_{facet}"].mean(),
                "negative_share_when_mentioned": group.loc[facet_hits, "neg_heavy"].mean() if facet_hits.any() else np.nan,
                "mentions": int(facet_hits.sum()),
                "reviews": int(len(group)),
                "days_since_last_mention": (TODAY - last_mention).days if pd.notna(last_mention) else 9999,
            }
        )

gap_df = pd.DataFrame(rows)
gap_df["gap_type"] = np.select(
    [
        (gap_df["listing_mentions_facet"] == 1) & (gap_df["review_mention_rate_lex"] < 0.05),
        (gap_df["listing_mentions_facet"] == 0) & (gap_df["review_mention_rate_lex"] >= 0.10),
        (gap_df["days_since_last_mention"] > 365) & (gap_df["mentions"] > 0),
        gap_df["negative_share_when_mentioned"].fillna(0) >= 0.50,
    ],
    [
        "listing talks about it, reviews rarely do",
        "reviews talk about it, listing barely does",
        "coverage is stale in reviews",
        "review sentiment is frequently negative",
    ],
    default="healthy_or_neutral",
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
coverage_gap = (
    gap_df.groupby("facet")[["listing_mentions_facet", "review_mention_rate_lex"]]
    .mean()
    .assign(review_mention_rate_lex=lambda df: df["review_mention_rate_lex"] * 100,
            listing_mentions_facet=lambda df: df["listing_mentions_facet"] * 100)
    .sort_values("review_mention_rate_lex")
)
coverage_gap.plot(kind="barh", ax=axes[0], color=["#1d4ed8", "#f97316"])
axes[0].set_title("Listing coverage vs lexical review coverage")
axes[0].set_xlabel("average % across properties")
axes[0].legend(["listing has facet text", "review mention rate"])

stale_pivot = gap_df.pivot(index="eg_property_id", columns="facet", values="days_since_last_mention").clip(upper=720)
sns.heatmap(stale_pivot, cmap="RdYlGn_r", ax=axes[1], cbar_kws={"label": "days since last mention (capped at 720)"})
axes[1].set_title("Lexical staleness by property and facet")
axes[1].set_xlabel("facet")
axes[1].set_ylabel("property")

plt.tight_layout()

gap_df.sort_values(["gap_type", "review_mention_rate_lex", "days_since_last_mention"], ascending=[True, True, False]).head(20)

In [ ]:
review_vecs, listing_semantic, proto_lookup, semantic_scores, semantic_hits, semantic_thresholds = fit_semantic_space(rev, listing_docs)
for facet in FACETS:
    rev[f"sem_{facet}"] = semantic_hits[facet].values

semantic_comparison = []
for facet in FACETS:
    lexical = rev[f"lex_{facet}"].astype(int)
    semantic = rev[f"sem_{facet}"].astype(int)
    both = int(((lexical == 1) & (semantic == 1)).sum())
    union = int(((lexical == 1) | (semantic == 1)).sum())
    semantic_comparison.append(
        {
            "facet": facet,
            "lexical_hits": int(lexical.sum()),
            "semantic_hits": int(semantic.sum()),
            "semantic_only": int(((semantic == 1) & (lexical == 0)).sum()),
            "lexical_only": int(((semantic == 0) & (lexical == 1)).sum()),
            "jaccard": both / union if union else 0.0,
            "threshold": semantic_thresholds[facet],
        }
    )
semantic_comparison = pd.DataFrame(semantic_comparison).sort_values("semantic_hits")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].barh(semantic_comparison["facet"], semantic_comparison["lexical_hits"], color="#f59e0b", alpha=0.85, label="lexical")
axes[0].barh(semantic_comparison["facet"], semantic_comparison["semantic_hits"], color="#2563eb", alpha=0.60, label="semantic")
axes[0].set_title("Lexical vs semantic coverage")
axes[0].set_xlabel("number of reviews classified")
axes[0].legend()

score_box = semantic_scores.copy()
score_box.columns = [f.replace("amenities_", "") for f in score_box.columns]
sns.boxplot(data=score_box, orient="h", ax=axes[1], color="#38bdf8")
axes[1].set_title("Semantic score distributions by facet")
axes[1].set_xlabel("max prototype cosine in LSA space")

plt.tight_layout()

semantic_gap_rows = []
for pid, group in rev.groupby("eg_property_id"):
    row_index = group.index.to_numpy()
    review_vectors = review_vecs[row_index]
    for facet in FACETS:
        listing_vec = listing_semantic.get(pid, {}).get(facet)
        if listing_vec is None:
            listing_mean_cos = np.nan
        else:
            listing_mean_cos = float(cosine_similarity(review_vectors, listing_vec.reshape(1, -1)).mean())
        semantic_gap_rows.append(
            {
                "eg_property_id": pid,
                "facet": facet,
                "semantic_review_mention_rate": group[f"sem_{facet}"].mean(),
                "lexical_review_mention_rate": group[f"lex_{facet}"].mean(),
                "listing_has_facet_text": int(listing_presence.loc[pid, facet]),
                "avg_semantic_score": float(semantic_scores.loc[group.index, facet].mean()),
                "listing_review_mean_cosine": listing_mean_cos,
                "semantic_negative_share": group.loc[group[f"sem_{facet}"] == 1, "neg_heavy"].mean() if group[f"sem_{facet}"].sum() else np.nan,
            }
        )
semantic_gap_df = pd.DataFrame(semantic_gap_rows)
semantic_gap_df["gap_rank"] = (
    1.5 * semantic_gap_df["listing_has_facet_text"]
    - semantic_gap_df["semantic_review_mention_rate"]
    - semantic_gap_df["listing_review_mean_cosine"].fillna(0)
    + semantic_gap_df["semantic_negative_share"].fillna(0)
)

semantic_comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
semantic_pivot = semantic_gap_df.pivot(index="eg_property_id", columns="facet", values="semantic_review_mention_rate")
sns.heatmap(semantic_pivot, cmap="magma", ax=axes[0], cbar_kws={"label": "semantic mention rate"})
axes[0].set_title("Property x facet semantic coverage")
axes[0].set_xlabel("facet")
axes[0].set_ylabel("property")

cos_pivot = semantic_gap_df.pivot(index="eg_property_id", columns="facet", values="listing_review_mean_cosine")
sns.heatmap(cos_pivot, cmap="RdYlGn", ax=axes[1], vmin=0.0, vmax=0.8, cbar_kws={"label": "listing-review cosine"})
axes[1].set_title("Listing text vs review semantics")
axes[1].set_xlabel("facet")
axes[1].set_ylabel("property")

plt.tight_layout()

semantic_gap_df.sort_values("gap_rank", ascending=False).head(20)

In [ ]:
n_clusters = min(12, max(4, rev["eg_property_id"].nunique()))
cluster_model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
rev["topic_cluster"] = cluster_model.fit_predict(review_vecs)

cluster_preview = []
for cluster_id in sorted(rev["topic_cluster"].unique()):
    cluster_idx = rev.index[rev["topic_cluster"] == cluster_id]
    centroid = cluster_model.cluster_centers_[cluster_id].reshape(1, -1)
    scores = cosine_similarity(review_vecs[cluster_idx], centroid).ravel()
    top_rows = rev.loc[cluster_idx[np.argsort(scores)[-3:][::-1]], ["eg_property_id", "full_text"]]
    cluster_preview.append(
        {
            "cluster": cluster_id,
            "size": int(len(cluster_idx)),
            "sample_snippets": [text[:180] for text in top_rows["full_text"].tolist()],
        }
    )
cluster_preview = pd.DataFrame(cluster_preview).sort_values("size", ascending=False)

dominant_semantic_facet = semantic_scores.idxmax(axis=1)
cluster_facet_mix = pd.crosstab(rev["topic_cluster"], dominant_semantic_facet, normalize="index")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
cluster_sizes = rev["topic_cluster"].value_counts().sort_index()
axes[0].bar(cluster_sizes.index.astype(str), cluster_sizes.values, color="#6366f1")
axes[0].set_title("Unsupervised topic cluster sizes")
axes[0].set_xlabel("cluster")
axes[0].set_ylabel("reviews")

sns.heatmap(cluster_facet_mix, cmap="Blues", ax=axes[1], cbar_kws={"label": "share of cluster"})
axes[1].set_title("Cluster mix by dominant semantic facet")
axes[1].set_xlabel("dominant facet")
axes[1].set_ylabel("cluster")

plt.tight_layout()
cluster_preview.head(10)

In [ ]:
ml_rows = []

if "rating_overall" in rev.columns:
    ml_binary = rev[(rev["rating_overall"].fillna(0) > 0) & (rev["full_text"].str.len() >= 20)].copy()
    ml_binary["low_rating_flag"] = (ml_binary["rating_overall"] <= 3).astype(int)

    if ml_binary["low_rating_flag"].sum() >= 25 and ml_binary["eg_property_id"].nunique() >= 3:
        random_binary = evaluate_binary(
            ml_binary,
            "low_rating_flag",
            StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
            use_grouped=False,
        ).assign(task="low_rating_flag", evaluation="random_cv")
        grouped_binary = evaluate_binary(
            ml_binary,
            "low_rating_flag",
            GroupKFold(n_splits=min(3, ml_binary["eg_property_id"].nunique())),
            use_grouped=True,
        ).assign(task="low_rating_flag", evaluation="group_property_cv")
        ml_rows.extend([random_binary, grouped_binary])

    ml_reg = rev[(rev["rating_overall"].fillna(0) > 0) & (rev["full_text"].str.len() >= 20)].copy()
    if len(ml_reg) >= 100 and ml_reg["eg_property_id"].nunique() >= 3:
        random_reg = evaluate_regression(
            ml_reg,
            "rating_overall",
            GroupKFold(n_splits=min(3, ml_reg["eg_property_id"].nunique())),
            use_grouped=True,
        ).assign(task="overall_rating", evaluation="group_property_cv")
        ml_rows.append(random_reg)

ml_results = pd.concat(ml_rows, ignore_index=True) if ml_rows else pd.DataFrame()

if not ml_results.empty:
    display(ml_results)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    binary_plot = ml_results.dropna(subset=["roc_auc"]) if "roc_auc" in ml_results.columns else pd.DataFrame()
    if not binary_plot.empty:
        sns.barplot(data=binary_plot, x="roc_auc", y="evaluation", hue="model", ax=axes[0])
        axes[0].set_title("Low-rating classification feasibility")
        axes[0].set_xlim(0.45, 1.0)
    else:
        axes[0].axis("off")

    reg_plot = ml_results.dropna(subset=["mae"]) if "mae" in ml_results.columns else pd.DataFrame()
    if not reg_plot.empty:
        sns.barplot(data=reg_plot, x="mae", y="evaluation", hue="model", ax=axes[1])
        axes[1].set_title("Overall rating regression feasibility")
    else:
        axes[1].axis("off")

    plt.tight_layout()
else:
    display(Markdown("Not enough labeled rows for the lightweight ML checks."))

In [ ]:
summary = pd.concat(
    [
        gap_df.assign(method="lexical")[
            [
                "eg_property_id",
                "facet",
                "listing_mentions_facet",
                "review_mention_rate_lex",
                "negative_share_when_mentioned",
                "days_since_last_mention",
                "gap_type",
                "method",
            ]
        ].rename(columns={"review_mention_rate_lex": "review_mention_rate", "negative_share_when_mentioned": "negative_share"}),
        semantic_gap_df.assign(method="semantic")[
            [
                "eg_property_id",
                "facet",
                "listing_has_facet_text",
                "semantic_review_mention_rate",
                "semantic_negative_share",
                "listing_review_mean_cosine",
                "gap_rank",
                "method",
            ]
        ].rename(columns={
            "listing_has_facet_text": "listing_mentions_facet",
            "semantic_review_mention_rate": "review_mention_rate",
            "semantic_negative_share": "negative_share",
        }),
    ],
    ignore_index=True,
    sort=False,
)

lexical_findings = (
    gap_df[gap_df["gap_type"] != "healthy_or_neutral"]
    .sort_values(["gap_type", "review_mention_rate_lex", "negative_share_when_mentioned"], ascending=[True, True, False])
    .head(10)
)
semantic_findings = semantic_gap_df.sort_values("gap_rank", ascending=False).head(10)

print("Top lexical findings")
display(lexical_findings)
print("Top semantic findings")
display(semantic_findings)

facet_rollup = pd.DataFrame({
    "listing_coverage_pct": listing_presence.mean() * 100,
    "lexical_review_coverage_pct": rev[[f"lex_{facet}" for facet in FACETS]].mean().values * 100,
    "semantic_review_coverage_pct": rev[[f"sem_{facet}" for facet in FACETS]].mean().values * 100,
}).round(2)
facet_rollup.index = FACETS
facet_rollup.sort_values("semantic_review_coverage_pct")

## How To Use The Output

Use the notebook in this order:

1. Start with the profile and rating coverage cells to understand the raw data limits.
2. Use the lexical coverage and staleness charts to find obvious missing coverage and dated topics.
3. Use the semantic section to catch less literal mentions that regex misses.
4. Use the clustering section to discover topics that should become first-class facets later.
5. Use the ML section as a feasibility check, not as a production model recommendation.

A good next iteration would be to replace the local LSA semantic space with embedding models, then validate the top semantic-only findings manually and promote the best new patterns into the lexical rules or product taxonomy.